# Sampling and Monte Carlo Methods

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 03.08 |
| Time | ~90 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/03_Probability_and_Statistics/09_sampling_and_monte_carlo.ipynb)

---

## 🎯 Learning Objective

Learn rejection sampling, importance sampling, MCMC/Gibbs sampling, and Hamiltonian Monte Carlo (HMC) as ways to draw samples from distributions that are hard to sample directly.

---

## 📐 Theory

Most interesting distributions can't be sampled directly by inverting their CDF (`03.02`) —
either the CDF has no closed form, or the distribution is only known up to an unknown
normalizing constant (a very common situation for Bayesian posteriors, `03.05`). This notebook
covers the general-purpose sampling algorithms that handle exactly this case.

### Rejection sampling

To sample from a target density $f(x)$ that's hard to sample directly, pick a **proposal**
distribution $q(x)$ that's easy to sample from and a constant $M$ such that $f(x) \le M\,q(x)$
for every $x$ (an *envelope*). Then: draw $x\sim q$, draw $u\sim\text{Uniform}(0,1)$, and
**accept** $x$ only if $u \le f(x)/(M q(x))$; otherwise reject and try again. Accepted samples
are exactly distributed as $f$ — accepting with probability proportional to how much of the
envelope $f$ fills at that $x$ corrects the proposal's shape into the target's shape. The
**acceptance rate** is $1/M$, so a loose envelope (large $M$) wastes most proposals — a central
weakness in high dimensions, where a tight envelope becomes almost impossible to construct.

### Importance sampling

Rather than reshaping a proposal into a target by discarding samples, **importance sampling**
keeps every sample and *reweights* it. To estimate $\mathbb{E}_f[h(X)] = \int h(x) f(x)\,dx$
using only samples from a proposal $q$:

$$\mathbb{E}_f[h(X)] = \int h(x) \frac{f(x)}{q(x)} q(x)\,dx = \mathbb{E}_q\left[h(X)\,w(X)\right], \qquad w(x) = \frac{f(x)}{q(x)},$$

where $w(x)$ is the **importance weight**. Draw $x_i\sim q$, and estimate the expectation as
$\frac{1}{n}\sum_i h(x_i)w(x_i)$ — no samples wasted, but the estimator's variance blows up
whenever $q$ is small where $f$ is large (a few huge weights dominate the sum), so choosing $q$
close to $f$ still matters enormously.

### Markov Chain Monte Carlo (MCMC)

When even evaluating $f$ up to its normalizing constant is all you have (the norm situation for
posteriors $p(\theta\mid D)\propto P(D\mid\theta)P(\theta)$, where the denominator $P(D)$ is an
intractable integral), **MCMC** builds a Markov chain whose *stationary distribution* is the
target $f$, then simulates it and collects visited states as (correlated) samples.
**Metropolis-Hastings** is the general recipe: from current state $x_t$, propose $x'\sim
q(\cdot\mid x_t)$, and accept it as $x_{t+1}$ with probability

$$\alpha = \min\left(1, \frac{f(x')\,q(x_t\mid x')}{f(x_t)\,q(x'\mid x_t)}\right);$$

otherwise stay at $x_{t+1}=x_t$. Because $f$ only ever appears as a *ratio* $f(x')/f(x_t)$, any
unknown normalizing constant cancels exactly — the algorithm never needs to know it.

**Why $f$ is stationary: detailed balance.** The accept/reject rule is constructed so the
chain's transition kernel $K(x\to x')$ satisfies **detailed balance** (reversibility) with
respect to $f$:

$$f(x)\,K(x\to x') = f(x')\,K(x'\to x) \qquad \text{for every pair } x,x'.$$

Summing/integrating both sides over $x$ shows $f$ is a stationary distribution of the chain
($\sum_x f(x) K(x\to x') = f(x')$ for every $x'$) — detailed balance is *sufficient* for
stationarity, and it's exactly what the $\min(1, \cdot)$ acceptance ratio was reverse-engineered
to guarantee: plug $K$ into the equation above and the proposal density $q$ and the
acceptance probabilities cancel it out algebraically, for *any* proposal $q$.

### Gibbs sampling

**Gibbs sampling** is a special case of Metropolis-Hastings for multivariate targets, with an
acceptance probability that's always exactly 1. To sample $(X,Y)\sim f(x,y)$, alternate:
$x_{t+1}\sim f(x\mid y_t)$, then $y_{t+1}\sim f(y\mid x_{t+1})$ — cycling through each variable's
full conditional (`03.04`) with every other variable held fixed. It needs those conditionals to
be directly sampleable, but when they are (common for conjugate Bayesian models, `03.05`-`03.06`),
it needs no proposal tuning at all.

### Hamiltonian Monte Carlo: using gradients to explore faster

Random-walk Metropolis-Hastings proposes the next state by adding **noise** to the current
state — a blind perturbation with no sense of direction. In $d$ dimensions that blind walk
needs $O(d^2)$ steps to traverse the typical set of the target, because each step's direction
is random. **Hamiltonian Monte Carlo (HMC)** exploits *gradient information* to make proposals
that follow the target's geometry, the same way momentum (`04.04`) accelerates gradient descent.

**Setup.** Augment the parameter $\theta\in\mathbb{R}^d$ with a **momentum** variable
$p\in\mathbb{R}^d$, drawn fresh each iteration from $p\sim\mathcal{N}(0,M)$ where $M$ is a
**mass matrix** (often $M=I$). Define the **Hamiltonian**

$$H(\theta,p) \;=\; \underbrace{-\log f(\theta)}_{\text{potential energy }U(\theta)} \;+\; \underbrace{\tfrac{1}{2}\,p^\top M^{-1}\,p}_{\text{kinetic energy }K(p)},$$

so that the joint density $\exp(-H(\theta,p))$ factorizes as $f(\theta)\,\mathcal{N}(p\mid 0,M)$
— marginalizing out $p$ recovers the target $f(\theta)$ exactly.

**Leapfrog integrator.** HMC simulates Hamilton's equations
$\dot\theta = M^{-1}p,\;\dot p = -\nabla_\theta U(\theta)$ for $L$ steps of size $\epsilon$
using the **leapfrog** scheme — a symplectic (volume-preserving, time-reversible) integrator
that keeps the numerical Hamiltonian close to the true one:

$$\text{half-step momentum:}\quad p \;\leftarrow\; p \;-\; \tfrac{\epsilon}{2}\,\nabla_\theta U(\theta),$$
$$\text{full-step position:}\quad \theta \;\leftarrow\; \theta \;+\; \epsilon\,M^{-1}\,p,$$
$$\text{half-step momentum:}\quad p \;\leftarrow\; p \;-\; \tfrac{\epsilon}{2}\,\nabla_\theta U(\theta).$$

Repeat these three lines $L$ times. The resulting $(\theta',p')$ is accepted with Metropolis
probability $\min\!\big(1,\,\exp(-H(\theta',p')+H(\theta,p))\big)$ — because leapfrog is
nearly energy-conserving, this acceptance rate is typically **very high** (> 90%), even in high
dimensions where random-walk MH would have acceptance rates near zero.

**Why HMC matters:** the leapfrog trajectory follows the target's contours instead of
random-walking across them. This reduces the number of gradient evaluations needed to produce
one independent sample from $O(d^2)$ (random-walk MH) to $O(d^{5/4})$ — a dramatic
improvement for the high-dimensional posteriors common in Bayesian deep learning.

**NUTS (No-U-Turn Sampler)** automates the choice of $L$ and $\epsilon$ by running leapfrog
until the trajectory starts to "double back" (the U-turn criterion), eliminating HMC's two
most sensitive tuning parameters. NUTS is the default MCMC algorithm in **Stan**, **PyMC**,
and **NumPyro** — when a practitioner says "I ran MCMC," they almost certainly mean NUTS-HMC.

### Why any of this converges: stationarity is not enough, you also need ergodicity

All of these are **Monte Carlo methods**: they replace an intractable integral/expectation with
a sample average, justified by the **Law of Large Numbers**. For iid samples this is immediate;
for MCMC it needs one more ingredient. Detailed balance only shows $f$ is *a* stationary
distribution of the chain — it does not by itself guarantee the chain's long-run visit
frequencies converge to $f$ *starting from an arbitrary initial state*, nor that $f$ is the
*only* stationary distribution. The condition that closes this gap is **ergodicity**: the chain
must be **irreducible** (every state reachable from every other state in finitely many steps —
so the chain can't stay trapped in one region of the support forever) and **aperiodic** (it
doesn't cycle deterministically through a fixed set of states). Under detailed balance +
ergodicity, the ergodic theorem guarantees $\frac{1}{n}\sum_{t=1}^n h(x_t) \to \mathbb{E}_f[h(X)]$
regardless of the starting state, and this — not "run it long enough" on its own — is what makes
"long enough" mean anything: a chain that isn't irreducible (e.g. a proposal step size so small
it essentially never crosses between two well-separated modes of a multimodal $f$) can run for an
arbitrarily long time while still only ever sampling from one mode, arbitrarily far from $f$.
That failure mode is demonstrated numerically below. MCMC's added practical subtlety, even when
the chain genuinely is ergodic, is that consecutive samples are correlated, so it needs a
"burn-in" period and yields a smaller *effective* sample size than its raw count.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Rejection sampling made visible: proposal samples scattered under an envelope, colored by
accept/reject, so the accepted points visibly trace out the target density's shape.

In [ ]:
# Rejection sampling from a bimodal target f(x) using a scaled Uniform(-4,4) envelope: every
# proposed point is a dot, colored by whether the acceptance test kept it.
rng = np.random.default_rng(0)

def target_density(x):
    # An unnormalized bimodal density -- deliberately NOT a named scipy distribution, to make
    # the point that rejection sampling only needs to EVALUATE f, not know its normalizer or CDF
    return 0.6 * np.exp(-0.5 * ((x - 2) / 0.7) ** 2) + 0.4 * np.exp(-0.5 * ((x + 2) / 1.0) ** 2)

x_range = np.linspace(-6, 6, 500)
M = 1.05 * target_density(x_range).max()  # envelope height: proposal density * M must dominate f everywhere
proposal_low, proposal_high = -6, 6
envelope_height = M / (proposal_high - proposal_low)  # since Uniform's density is 1/(high-low)

n_proposals = 3000
proposals_x = rng.uniform(proposal_low, proposal_high, n_proposals)
proposals_u = rng.uniform(0, 1, n_proposals) * envelope_height  # height under the flat envelope
accept_mask = proposals_u <= target_density(proposals_x) / (proposal_high - proposal_low)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(proposals_x[~accept_mask], proposals_u[~accept_mask], s=4, alpha=0.3, color="gray", label="rejected")
axes[0].scatter(proposals_x[accept_mask], proposals_u[accept_mask], s=4, alpha=0.6, color="#C44E52", label="accepted")
axes[0].plot(x_range, target_density(x_range) / (proposal_high - proposal_low), color="#4C72B0", lw=2, label="f(x) [rescaled]")
axes[0].axhline(envelope_height, color="black", ls="--", lw=1, label="envelope M*q(x)")
axes[0].set_title("Rejection sampling: accept iff the proposal\nfalls UNDER the target curve"); axes[0].legend(fontsize=8)

axes[1].hist(proposals_x[accept_mask], bins=50, density=True, color="#C44E52", alpha=0.6, label="accepted samples")


def trapz_1d(values, x):
    # Hand-rolled trapezoidal rule -- avoids relying on np.trapz/np.trapezoid across numpy versions.
    widths = np.diff(x)
    return np.sum(widths * (values[:-1] + values[1:]) / 2)


normalizer = trapz_1d(target_density(x_range), x_range)
axes[1].plot(x_range, target_density(x_range) / normalizer, "k--", lw=2, label="true (normalized) f(x)")
axes[1].set_title("Accepted samples reproduce the target density"); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Acceptance rate: {accept_mask.mean():.3f}  (theoretical ~ 1/M-ish, set by how tight the envelope is)")
print(f"Number of accepted samples: {accept_mask.sum()} out of {n_proposals} proposals")

## 🐍 Implementation from Scratch

We implement importance sampling for an expectation, Metropolis-Hastings MCMC for a 2D target
with an unknown normalizing constant, Gibbs sampling for a bivariate Gaussian using its
known conditional distributions, and Hamiltonian Monte Carlo (HMC) with a leapfrog integrator
— comparing every result against ground truth.

In [ ]:
# --- Importance sampling: estimate E_f[X] for f=Normal(3,1), using ONLY samples from a
# different proposal q=Normal(0,2), reweighted by w(x) = f(x)/q(x) ---
rng = np.random.default_rng(0)
f_dist, q_dist = stats.norm(3, 1), stats.norm(0, 2)   # TARGET vs. PROPOSAL

n_samples = 20_000
samples_q = q_dist.rvs(size=n_samples, random_state=0)
# Log-space weights, normalized via the same max-subtraction trick as a stable softmax
# (recall 06.02): computing f_dist.pdf(x)/q_dist.pdf(x) directly divides two densities that
# could each individually underflow to exactly 0 far in either distribution's tails (0/0=nan),
# or, for a target with heavier tails than the proposal, produce a weight that overflows.
# Subtracting the max LOG-weight before exponentiating avoids both failure modes exactly the
# way it does for softmax, while leaving the self-normalized estimate mathematically identical
# (multiplying every weight by the same constant e^{-max} cancels in the ratio below).
log_weights = f_dist.logpdf(samples_q) - q_dist.logpdf(samples_q)
weights = np.exp(log_weights - log_weights.max())
importance_estimate = np.sum(samples_q * weights) / np.sum(weights)  # self-normalized IS

print(f"True E_f[X] = {f_dist.mean():.4f}")
print(f"Importance sampling estimate (samples drawn from a DIFFERENT distribution): {importance_estimate:.4f}")
print(f"Effective sample size: {np.sum(weights)**2 / np.sum(weights**2):.1f} out of {n_samples}")

# --- Metropolis-Hastings MCMC: sample a target known only up to an unknown constant ---
def log_unnormalized_target(xy):
    # Correlated 2D Gaussian shape, scaled by an ARBITRARY constant (log(37.2) added at the
    # end): MCMC only ever uses the RATIO target(x')/target(x), so an unknown normalizer
    # cancels automatically -- working in LOG space from the start (rather than exponentiating
    # here and dividing raw densities later) is what keeps that cancellation numerically safe.
    x, y, rho = xy[0], xy[1], 0.8
    return np.log(37.2) - (x**2 - 2 * rho * x * y + y**2) / (2 * (1 - rho**2))

def metropolis_hastings(log_target_unnorm, init, n_steps, proposal_std, rng):
    # Accept/reject via the LOG-density difference, not a raw ratio of densities: computing
    # target(proposal)/target(current) directly (as an earlier version of this function did)
    # divides two np.exp(...) outputs, and for any target whose log-density spans more than
    # ~700 in magnitude -- trivially true for realistic Bayesian posteriors over more than a
    # handful of data points, where log-likelihood scales with N -- BOTH numerator and
    # denominator can individually overflow/underflow, producing a 0/0 = nan ratio (chain
    # corrupts silently) or 0/positive = 0 (chain freezes, looking merely "stuck", not
    # obviously broken). log_target_unnorm(proposal) - log_target_unnorm(current) never has
    # this problem: subtracting two logs is exactly as safe as subtracting two ordinary
    # numbers, and the one exponential taken afterward (via np.exp, capped by the min(0, ...))
    # always has a non-positive argument, so it can never overflow.
    current = np.array(init, dtype=float)
    samples, n_accepted = np.empty((n_steps, 2)), 0
    for t in range(n_steps):
        proposal = current + rng.normal(scale=proposal_std, size=2)  # symmetric -> q cancels in alpha
        log_accept_ratio = log_target_unnorm(proposal) - log_target_unnorm(current)
        if np.log(rng.uniform()) < min(0.0, log_accept_ratio):
            current = proposal
            n_accepted += 1
        samples[t] = current
    return samples, n_accepted / n_steps

mh_samples, accept_rate = metropolis_hastings(log_unnormalized_target, init=[0, 0], n_steps=20_000,
                                               proposal_std=0.6, rng=rng)
burn_in = 2000
mh_kept = mh_samples[burn_in:]
print(f"\nMCMC acceptance rate: {accept_rate:.3f}")
print(f"MCMC sample mean (should be ~[0,0]): {mh_kept.mean(axis=0).round(3)}")
print(f"MCMC sample correlation (should be ~0.8): {np.corrcoef(mh_kept.T)[0,1]:.3f}")

# --- Regression test: a target whose LOG-density spans >700 (a stand-in for a realistic
# posterior over many i.i.d. data points) makes the OLD raw-ratio acceptance rule fail --
# both target_unnorm(proposal) and target_unnorm(current) individually overflow/underflow --
# while the log-space rule stays exact. ---
def raw_ratio_accept_step(log_target, current, proposal):
    """The OLD, unguarded pattern: exponentiate each log-density separately, then divide."""
    with np.errstate(over='ignore', invalid='ignore', divide='ignore'):
        target_current = np.exp(log_target(current))
        target_proposal = np.exp(log_target(proposal))
        return target_proposal / target_current

def wide_range_log_target(xy):
    # log-density scaled by 1000x relative to log_unnormalized_target -- same SHAPE (so the
    # correct accept/reject decision is identical), but now exp() of the raw log-density
    # routinely exceeds float64's overflow threshold (~709) for typical proposal/current pairs.
    x, y, rho = xy[0], xy[1], 0.8
    return 1000.0 * (-(x**2 - 2 * rho * x * y + y**2) / (2 * (1 - rho**2)))

current_pt, proposal_pt = np.array([3.0, 3.0]), np.array([3.2, 2.7])
log_ratio_correct = wide_range_log_target(proposal_pt) - wide_range_log_target(current_pt)
with np.errstate(over='ignore', invalid='ignore', divide='ignore'):
    raw_ratio = raw_ratio_accept_step(wide_range_log_target, current_pt, proposal_pt)

print(f"\n--- Regression test: MH acceptance on a target with a WIDE log-density range ---")
print(f"True log(accept ratio) = {log_ratio_correct:.4f}  (a small, ordinary, perfectly usable number)")
print(f"OLD raw-ratio approach: target(proposal)/target(current) = {raw_ratio}  "
      f"({'nan -- BROKEN: both densities underflowed to exactly 0, giving 0/0' if not np.isfinite(raw_ratio) else 'finite'})")
print(f"NEW log-space approach never has this problem: log_target(proposal) - log_target(current)")
print(f"is an ordinary subtraction of two finite numbers, however extreme each is individually.")
assert not np.isfinite(raw_ratio), "expected the old raw-ratio approach to break (produce nan) on this wide-range target"
assert np.isfinite(log_ratio_correct), "the log-space approach must stay finite on the same input"

# --- Gibbs sampling: full conditionals of a bivariate Gaussian are KNOWN Gaussians, so every
# step samples EXACTLY (acceptance probability always 1, no rejection needed) ---
def gibbs_bivariate_gaussian(rho, n_steps, rng):
    x, y, cond_std = 0.0, 0.0, np.sqrt(1 - rho**2)
    samples = np.empty((n_steps, 2))
    for t in range(n_steps):
        x = rng.normal(rho * y, cond_std)   # x | y ~ N(rho*y, 1-rho^2)
        y = rng.normal(rho * x, cond_std)   # y | x ~ N(rho*x, 1-rho^2)
        samples[t] = (x, y)
    return samples

gibbs_kept = gibbs_bivariate_gaussian(rho=0.8, n_steps=20_000, rng=rng)[burn_in:]
print(f"\nGibbs sample mean (should be ~[0,0]):      {gibbs_kept.mean(axis=0).round(3)}")
print(f"Gibbs sample correlation (should be ~0.8): {np.corrcoef(gibbs_kept.T)[0,1]:.3f}")

# --- Ergodicity, made concrete: detailed balance alone does NOT guarantee convergence to f.
# Two Metropolis-Hastings chains below use the IDENTICAL accept/reject rule (so BOTH satisfy
# detailed balance w.r.t. the same target f), differing only in proposal step size. f is a
# symmetric two-mode target (equal-weight Gaussians at x=-4 and x=+4), so its TRUE mean is 0
# and exactly half its mass lies on each side of 0 -- known ground truth, not estimated.
def log_bimodal_1d_unnorm(x):
    # log(exp(a) + exp(b)), computed via the log-sum-exp trick (recall 06.02) rather than
    # np.log(np.exp(a) + np.exp(b)) directly -- the naive two-step version would reintroduce
    # exactly the overflow risk this fix is removing, just one function layer further in.
    a = -0.5 * (x - 4.0) ** 2
    b = -0.5 * (x + 4.0) ** 2
    m = np.maximum(a, b)
    return m + np.log(np.exp(a - m) + np.exp(b - m))  # equal-weight modes at +/-4

def metropolis_hastings_1d(log_target_unnorm, init, n_steps, proposal_std, rng):
    # Same log-space acceptance rule as `metropolis_hastings` above, for the same reason:
    # a raw ratio of two np.exp(...) outputs can silently overflow/underflow/NaN for any
    # target whose log-density range exceeds float64's ~700 window, while the log-difference
    # form never has that problem.
    current, samples, n_accepted = float(init), np.empty(n_steps), 0
    for t in range(n_steps):
        proposal = current + rng.normal(scale=proposal_std)          # symmetric proposal -> q cancels
        log_accept_ratio = log_target_unnorm(proposal) - log_target_unnorm(current)
        if np.log(rng.uniform()) < min(0.0, log_accept_ratio):
            current = proposal
            n_accepted += 1
        samples[t] = current
    return samples, n_accepted / n_steps

true_mean_bimodal, true_frac_positive = 0.0, 0.5  # target is symmetric about 0 by construction
rng_erg = np.random.default_rng(7)
n_steps_erg = 20_000

# SAME detailed-balance-satisfying accept rule, SAME target, SAME start (deliberately at x=4,
# i.e. started ON one mode) -- only the proposal step size (hence how easily the chain can
# cross the low-density gap between modes) differs.
poorly_mixed, accept_poor = metropolis_hastings_1d(log_bimodal_1d_unnorm, init=4.0, n_steps=n_steps_erg,
                                                    proposal_std=0.5, rng=rng_erg)
well_mixed, accept_good = metropolis_hastings_1d(log_bimodal_1d_unnorm, init=4.0, n_steps=n_steps_erg,
                                                  proposal_std=4.0, rng=rng_erg)

print("--- Ergodicity check: detailed balance holds in BOTH chains (identical accept rule), ---")
print("--- but only the chain that can actually reach both modes converges to the true f ---")
for label, chain, acc in [("small step, std=0.5 (rarely crosses the gap between modes)", poorly_mixed, accept_poor),
                           ("large step, std=4.0 (crosses between modes often)", well_mixed, accept_good)]:
    frac_positive = np.mean(chain > 0)
    print(f"  {label}")
    print(f"    acceptance rate={acc:.3f}, sample mean={chain.mean():+.3f} (true={true_mean_bimodal}), "
          f"frac(x>0)={frac_positive:.3f} (true={true_frac_positive})")
print("\nBoth chains are reversible w.r.t. f by construction (detailed balance never fails here) --")
print("but the small-step chain is not IRREDUCIBLE within this many steps (it essentially never")
print("crosses the gap), so it stays near its x=4 start and its sample mean stays badly biased.")
print("Running it for MORE steps at this same proposal_std would not obviously fix this -- crossing")
print("the gap is a rare event either way. This is exactly why ergodicity (irreducibility +")
print("aperiodicity), not detailed balance alone and not just 'run it long enough', is the")
print("condition that actually justifies using a chain's samples as draws from the target f.")

In [ ]:
# --- Hamiltonian Monte Carlo from scratch: leapfrog integrator + full HMC sampler ---
# Target: 2D correlated Gaussian (same as MH above) — known ground truth for validation.
rng = np.random.default_rng(42)

def grad_neg_log_target(xy, rho=0.8):
    """Gradient of U(theta) = -log f(theta) for the 2D correlated Gaussian."""
    x, y = xy[0], xy[1]
    denom = 1 - rho**2
    grad_x = (x - rho * y) / denom
    grad_y = (y - rho * x) / denom
    return np.array([grad_x, grad_y])

def leapfrog(theta, p, grad_U, epsilon, L):
    """Leapfrog integrator: L steps of size epsilon.
    Three-step pattern per iteration: half-momentum, full-position, half-momentum.
    Symplectic (volume-preserving + time-reversible) — critical for HMC correctness."""
    theta = theta.copy()
    p = p.copy()
    # Half step for momentum at the start
    p -= 0.5 * epsilon * grad_U(theta)
    for i in range(L - 1):
        # Full step for position
        theta += epsilon * p
        # Full step for momentum
        p -= epsilon * grad_U(theta)
    # Final full step for position
    theta += epsilon * p
    # Half step for momentum at the end
    p -= 0.5 * epsilon * grad_U(theta)
    return theta, -p  # negate momentum for reversibility (not strictly needed with fresh p each iter)

def hamiltonian(theta, p, log_target_fn):
    """H(theta, p) = -log f(theta) + 0.5 * p^T p  (mass matrix M = I)."""
    U = -log_target_fn(theta)
    K = 0.5 * np.dot(p, p)
    return U + K

def hmc_sampler(log_target_fn, grad_U, init, n_steps, epsilon, L, rng):
    """Full HMC sampler with Metropolis accept/reject on the Hamiltonian."""
    d = len(init)
    current = np.array(init, dtype=float)
    samples = np.empty((n_steps, d))
    n_accepted = 0
    for t in range(n_steps):
        p_current = rng.standard_normal(d)  # fresh momentum ~ N(0, I)
        H_current = hamiltonian(current, p_current, log_target_fn)
        theta_prop, p_prop = leapfrog(current, p_current, grad_U, epsilon, L)
        H_prop = hamiltonian(theta_prop, p_prop, log_target_fn)
        # Metropolis accept/reject — in log space for safety
        if np.log(rng.uniform()) < H_current - H_prop:
            current = theta_prop
            n_accepted += 1
        samples[t] = current
    return samples, n_accepted / n_steps

# --- Run HMC and random-walk MH on the SAME 2D correlated Gaussian target ---
n_steps_compare = 5000
burn_in_compare = 500

# HMC: epsilon=0.15, L=25 leapfrog steps
hmc_samples, hmc_accept = hmc_sampler(
    log_unnormalized_target, grad_neg_log_target,
    init=[0.0, 0.0], n_steps=n_steps_compare, epsilon=0.15, L=25, rng=rng
)

# Random-walk MH (reuse the function defined above)
rng_mh = np.random.default_rng(42)
mh_samples_compare, mh_accept_compare = metropolis_hastings(
    log_unnormalized_target, init=[0, 0], n_steps=n_steps_compare,
    proposal_std=0.6, rng=rng_mh
)

hmc_kept = hmc_samples[burn_in_compare:]
mh_kept_compare = mh_samples_compare[burn_in_compare:]

print("=== HMC vs Random-Walk Metropolis-Hastings ===")
print(f"HMC  acceptance rate: {hmc_accept:.3f}")
print(f"MH   acceptance rate: {mh_accept_compare:.3f}")
print(f"HMC  sample mean: {hmc_kept.mean(axis=0).round(3)}, corr: {np.corrcoef(hmc_kept.T)[0,1]:.3f}")
print(f"MH   sample mean: {mh_kept_compare.mean(axis=0).round(3)}, corr: {np.corrcoef(mh_kept_compare.T)[0,1]:.3f}")

# --- Effective sample size (ESS) via autocorrelation ---
def ess_1d(chain, max_lag=500):
    """Effective sample size from autocorrelation (Geyer's initial positive sequence estimator)."""
    n = len(chain)
    chain_centered = chain - chain.mean()
    var = np.var(chain_centered)
    if var == 0:
        return 1.0
    acf = np.correlate(chain_centered, chain_centered, mode='full')[n-1:]  # one-sided
    acf = acf / (var * n)
    # Sum consecutive pairs; stop when a pair sum goes negative (Geyer's criterion)
    tau = 1.0
    for k in range(1, min(max_lag, n // 2), 2):
        pair_sum = acf[k] + acf[k + 1] if k + 1 < len(acf) else acf[k]
        if pair_sum < 0:
            break
        tau += 2 * pair_sum
    return n / tau

ess_hmc_x = ess_1d(hmc_kept[:, 0])
ess_hmc_y = ess_1d(hmc_kept[:, 1])
ess_mh_x  = ess_1d(mh_kept_compare[:, 0])
ess_mh_y  = ess_1d(mh_kept_compare[:, 1])

print(f"\nEffective sample size (x-component):")
print(f"  HMC: {ess_hmc_x:.0f} / {len(hmc_kept)} samples  ({100*ess_hmc_x/len(hmc_kept):.1f}% efficiency)")
print(f"  MH:  {ess_mh_x:.0f} / {len(mh_kept_compare)} samples  ({100*ess_mh_x/len(mh_kept_compare):.1f}% efficiency)")

# --- Visual comparison: trajectories and sample clouds ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: First 200 steps of each chain — HMC takes long strides, MH random-walks
n_show = 200
axes[0].plot(mh_samples_compare[:n_show, 0], mh_samples_compare[:n_show, 1],
             '-o', ms=2, alpha=0.5, color='#C44E52', label='MH trajectory')
axes[0].plot(hmc_samples[:n_show, 0], hmc_samples[:n_show, 1],
             '-o', ms=2, alpha=0.5, color='#4C72B0', label='HMC trajectory')
axes[0].set_title(f'First {n_show} steps: HMC strides vs MH crawls')
axes[0].legend(fontsize=8); axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
axes[0].set_xlim(-4, 4); axes[0].set_ylim(-4, 4)

# Panel 2: Full sample clouds
axes[1].scatter(mh_kept_compare[::3, 0], mh_kept_compare[::3, 1], s=2, alpha=0.3,
                color='#C44E52', label=f'MH (ESS_x≈{ess_mh_x:.0f})')
axes[1].scatter(hmc_kept[::3, 0], hmc_kept[::3, 1], s=2, alpha=0.3,
                color='#4C72B0', label=f'HMC (ESS_x≈{ess_hmc_x:.0f})')
axes[1].set_title('Sample clouds (post burn-in)'); axes[1].legend(fontsize=8)
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')

# Panel 3: Autocorrelation comparison — HMC decorrelates much faster
max_lag_plot = 100
for label, chain, color in [('MH', mh_kept_compare[:, 0], '#C44E52'),
                              ('HMC', hmc_kept[:, 0], '#4C72B0')]:
    n = len(chain)
    centered = chain - chain.mean()
    full_acf = np.correlate(centered, centered, mode='full')[n-1:]
    acf_normed = full_acf / full_acf[0]
    axes[2].plot(acf_normed[:max_lag_plot], label=label, color=color, lw=2)
axes[2].axhline(0, color='gray', ls='--', lw=0.8)
axes[2].set_title('Autocorrelation (x-component)')
axes[2].set_xlabel('Lag'); axes[2].set_ylabel('ACF')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\nHMC decorrelates in ~5-15 lags; MH needs ~50-100+. This is the efficiency gain.")
print("In high dimensions, this gap widens from a constant factor to a POLYNOMIAL improvement.")


## 🤖 Why This Matters for AI

Every one of these techniques shows up directly in modern ML: **variational inference** turns
an intractable Bayesian posterior into an optimization problem, but is validated against MCMC
ground truth; **Bayes by Backprop** (`03.11`) treats network weights as a distribution sampled
during training; and every time a language model generates text, it's **sampling from a
categorical distribution over tokens** — where temperature, top-k, and nucleus (top-p) sampling
are all just different ways of reshaping that distribution before drawing from it, demonstrated
concretely below.

---

In [ ]:
# LLM decoding strategies: temperature scaling, top-k, and nucleus (top-p) sampling all
# RESHAPE a categorical distribution (03.02, 03.06) over the vocabulary before drawing from it.
rng = np.random.default_rng(0)
vocab = [f"tok_{i}" for i in range(10)]
logits = np.array([4.0, 3.5, 2.0, 1.8, 1.0, 0.5, 0.2, -0.5, -1.0, -2.0])  # one skewed logit vector

def softmax(z):
    z = z - z.max()
    return np.exp(z) / np.exp(z).sum()

def temperature_scale(logits, T):
    # Dividing logits by T before softmax: T<1 sharpens the distribution (more confident,
    # closer to argmax), T>1 flattens it (more random) -- same logits, reshaped distribution.
    return softmax(logits / T)

def top_k_filter(probs, k):
    # Zero out everything except the k highest-probability tokens, then renormalize the rest.
    keep_idx = np.argsort(probs)[-k:]
    filtered = np.zeros_like(probs)
    filtered[keep_idx] = probs[keep_idx]
    return filtered / filtered.sum()

def nucleus_filter(probs, p):
    # Keep the SMALLEST set of highest-probability tokens whose cumulative mass exceeds p,
    # then renormalize -- the set size adapts to how "peaked" or "flat" the distribution is.
    order = np.argsort(probs)[::-1]
    cumulative = np.cumsum(probs[order])
    cutoff = np.searchsorted(cumulative, p) + 1
    keep_idx = order[:cutoff]
    filtered = np.zeros_like(probs)
    filtered[keep_idx] = probs[keep_idx]
    return filtered / filtered.sum(), cutoff

base_probs = softmax(logits)
print("Base softmax probabilities (T=1, no filtering):")
print(np.round(base_probs, 4))

print("\n--- Temperature scaling reshapes concentration, without changing WHICH token ranks highest ---")
for T in [0.3, 1.0, 2.0]:
    probs_T = temperature_scale(logits, T)
    entropy_T = -np.sum(probs_T * np.log(probs_T + 1e-12))  # a concentration summary, formalized in 05.01
    print(f"  T={T:.1f}: top prob={probs_T.max():.4f}, entropy={entropy_T:.4f}, "
          f"argmax token={vocab[np.argmax(probs_T)]}")
print("  (lower T -> lower entropy, more concentrated on the top token; higher T -> flatter, more random)")

print("\n--- top-k keeps a FIXED candidate count regardless of how peaked the distribution is ---")
for k in [1, 3, 10]:
    probs_k = top_k_filter(base_probs, k)
    n_nonzero = np.sum(probs_k > 0)
    print(f"  k={k}: kept {n_nonzero} tokens, renormalized probs sum to {probs_k.sum():.4f}")

print("\n--- nucleus (top-p) keeps a VARIABLE candidate count, adapting to the distribution shape ---")
for p in [0.5, 0.9, 0.99]:
    probs_p, n_kept = nucleus_filter(base_probs, p)
    print(f"  p={p}: kept {n_kept} tokens (adaptive, not fixed) to cover >= {p:.0%} of the mass")

print("\nSampling 5 tokens from the T=0.5, top-k=3 reshaped distribution:")
final_probs = top_k_filter(temperature_scale(logits, 0.5), k=3)
sampled_tokens = rng.choice(vocab, size=5, p=final_probs)
print(" ", list(sampled_tokens))

## 🏋️ Exercises

Seven levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For the target $f(x)=2x$ on $[0,1]$ (already a valid, normalized density — check
   $\int_0^1 2x\,dx=1$ by hand) and proposal $q(x)=1$ (Uniform$(0,1)$), find the smallest valid
   envelope constant $M$ such that $f(x)\le M\,q(x)$ for every $x\in[0,1]$, and state the
   resulting theoretical acceptance rate $1/M$. Verify by generating proposals and checking the
   empirical acceptance rate matches.

<details>
<summary>💡 Solution</summary>

Sanity check first: $\int_0^1 2x\,dx = [x^2]_0^1 = 1$, confirming $f$ is a valid density. Since
$q(x)=1$, the condition $f(x)\le M q(x)$ becomes $2x\le M$ for all $x\in[0,1]$; the tightest
(smallest) valid $M$ is the maximum of $2x$ on $[0,1]$, which occurs at $x=1$: $M=2$. The
theoretical acceptance rate is $1/M=0.5$. Simulating $200{,}000$ proposals from Uniform$(0,1)$
with this $M=2$ envelope gives an empirical acceptance rate very close to $0.5$ (a representative
run gives $\approx 0.498$), matching the theory.

</details>

### 💭 UNDERSTAND
2. The Theory section requires $f(x)\le M q(x)$ for *every* $x$. Suppose someone picks an $M$
   that's too small — smaller than the true maximum of $f(x)/q(x)$ over the support (e.g. using
   `M_factor=0.8` on the notebook's own bimodal `target_density`, instead of the `1.05` the
   notebook uses). Without running anything, predict what goes wrong with the accepted samples'
   distribution, then verify.

<details>
<summary>💡 Solution</summary>

If $M$ is too small, the envelope $Mq(x)$ dips *below* $f(x)$ somewhere — typically right at
$f$'s peak, where $f(x)/q(x)$ is largest. Near that region, the acceptance test
$u\le f(x)/(Mq(x))$ can have $f(x)/(Mq(x))>1$, meaning it's testing against a ratio that exceeds
$1$ — proposals there get accepted *too readily* relative to how a correctly-scaled envelope
would treat them, but crucially, the algorithm structurally cannot accept "more than all" of
the proposals landing there, so the region right around the peak ends up systematically
**undersampled** relative to its true density, rather than correctly over-represented. Verifying
with `M_factor=0.8` on `target_density`: the maximum ratio $f(x)/(Mq(x))$ comes out above $1$
(around $1.25$ in a representative run, confirming the envelope fails to dominate), and the
empirical density of accepted samples near the true peak ($x=2$) comes out visibly lower than
the true normalized density there (roughly $0.25$ empirical vs. $0.29$ true in a representative
run) — a real, systematic distortion, not sampling noise.

</details>

### ✏️ DERIVE
3. Prove that rejection sampling's accepted samples are exactly distributed as $f$. Using
   $P(\text{accept}\mid X=x) = f(x)/(Mq(x))$ (the acceptance rule) and $X\sim q$, derive the
   conditional density $P(x\mid\text{accept})$ via Bayes' theorem (`03.05`) applied to densities,
   and show it simplifies to exactly $f(x)$.

<details>
<summary>💡 Solution outline</summary>

The joint "density" of drawing $x$ and accepting it is $q(x)\cdot P(\text{accept}\mid X=x) =
q(x)\cdot\frac{f(x)}{Mq(x)} = \frac{f(x)}{M}$ — notice $q(x)$ cancels exactly. The overall
acceptance probability is obtained by integrating this joint density over all $x$:
$P(\text{accept}) = \int \frac{f(x)}{M}\,dx = \frac{1}{M}\int f(x)\,dx = \frac{1}{M}$ (since $f$,
being a valid density, integrates to exactly $1$). By the continuous analogue of Bayes' theorem,
the density of $x$ *given* acceptance is
$$P(x\mid\text{accept}) = \frac{\text{joint density of }(x,\text{accept})}{P(\text{accept})}
= \frac{f(x)/M}{1/M} = f(x),$$
exactly the target density — confirming that conditioning on acceptance exactly undoes the
distortion the proposal $q$ and the envelope $M$ introduced, regardless of what $q$ or $M$
(as long as $M$ is valid) were.

</details>

### 🐍 IMPLEMENT
4. Modify `metropolis_hastings` to track and return the FULL chain (not just the samples after
   burn-in), then plot the sample path of $x_t$ (the first coordinate) over the first 200 steps.

<details>
<summary>✅ How to know you're right</summary>

The returned array should have shape `(n_steps, 2)` with `n_steps` matching the argument passed
in (not `n_steps - burn_in`), and the first entries should visibly start near `init` (e.g. near
`[0, 0]`) before settling into fluctuating around the target's stationary behavior — plotting
`full_chain[:200, 0]` should show a discernible transient near the start (potentially still
influenced by initialization) followed by a longer stretch that looks like stationary noise
fluctuating around a roughly constant level. This is the empirical basis for choosing a
`burn_in` value: discard however many initial steps visibly show transient, non-representative
behavior before the chain looks like it's fluctuating around its long-run behavior.

</details>

### 🤖 APPLY
5. In the importance sampling cell, sweep the proposal's standard deviation `q_dist =
   stats.norm(0, q_std)` over `q_std in [0.5, 1.0, 2.0, 4.0]`, keeping the target
   `f_dist = stats.norm(3, 1)` fixed. For each, record the effective sample size (ESS) and the
   resulting estimate of $\mathbb{E}_f[X]$.

<details>
<summary>✅ What you should observe</summary>

ESS should be dramatically lower for the narrower proposals ($q\_std=0.5$: ESS around $37$ out
of $20{,}000$; $q\_std=1.0$: around $102$) than for the wider ones ($q\_std=2.0$: around $3600$;
$q\_std=4.0$: around $5150$) — because a proposal centered at $0$ with a small standard
deviation puts very little mass near the target's mean of $3$, forcing a handful of lucky
far-tail draws to carry almost all the importance weight. Correspondingly, the actual numerical
estimate of $\mathbb{E}_f[X]=3.0$ is noticeably worse for the low-ESS cases (e.g. an estimate
around $1.4$–$2.7$ for $q\_std=0.5$ or $1.0$, badly off) than for the high-ESS cases (estimates
around $2.99$–$3.00$ for $q\_std=2.0$ or $4.0$) — confirming that ESS is a genuine warning
signal for estimator reliability, not just a diagnostic number to report and ignore.

</details>

### 🤖 APPLY (HMC)
6. Run HMC on a **10-dimensional** correlated Gaussian target (covariance matrix with
   $\Sigma_{ij} = 0.9^{|i-j|}$, a common test case). Compare ESS per second to random-walk MH
   at the same target: run both samplers for the same wall-clock budget, compute ESS for each
   coordinate, and report the median ESS/second ratio (HMC ÷ MH). Use the `leapfrog` and
   `hmc_sampler` functions from the HMC cell above, with $\epsilon=0.1$, $L=20$.

<details>
<summary>✅ What you should observe</summary>

HMC's ESS/second should be **5–20× higher** than MH's, even though each HMC step is more
expensive (it evaluates the gradient $L$ times per proposal). The advantage comes from far
lower autocorrelation: HMC's ESS/step is so much higher that it more than compensates for
the per-step cost. In 10 dimensions this is a moderate win; in 100+ dimensions (common for
real Bayesian models) the gap becomes orders of magnitude, which is why NUTS-HMC is the
universal default in probabilistic programming frameworks.

</details>

### 🚀 CHALLENGE
7. Cross-check a Monte Carlo estimate against ground truth as a function of sample size, and
   confirm the gap shrinks as predicted — but investigate whether it *always* does. Using the
   notebook's own importance-sampling setup (`f_dist=norm(3,1)`), estimate $\mathbb{E}_f[X]$ at
   several sample sizes $n\in\{100, 1000, 10000, 100000\}$ with `q_dist=norm(0,2)`, repeating
   each sample size multiple times (e.g. 30 independent repeats) to get a stable average
   absolute error at each $n$. Plot the average absolute error vs. $n$ on a log-log plot and
   check whether it's consistent with the $O(1/\sqrt n)$ Monte Carlo convergence rate that also
   governs plain i.i.d. averaging (`03.03`). Then repeat the exact same experiment with
   `q_dist=norm(0,0.5)` (a proposal noticeably *narrower* than the target). Does the error at
   `q_std=0.5` shrink at anything close to the same rate as `q_std=2.0`, even out to
   $n=100{,}000$? Investigate why, by inspecting the raw (non-normalized) importance weights
   `np.exp(f_dist.logpdf(samples) - q_dist.logpdf(samples))` at both `q_std` values — specifically
   their maximum and variance as `n` grows.

<details>
<summary>🔍 What a strong answer covers</summary>

With `q_std=2.0`, average absolute error shrinks cleanly with $n$ (a representative run: roughly
$0.156$ at $n{=}100$ down to roughly $0.005$ at $n{=}100{,}000$, a fitted log-log slope of about
$-0.50$, matching $O(1/\sqrt n)$ almost exactly). With `q_std=0.5`, the error is far larger at
every $n$ *and* barely shrinks at all across the same four-decade range of $n$ (representative
run: roughly $1.98\to1.63\to1.39\to1.29$, a fitted slope of only about $-0.06$ — nowhere close to
$-0.5$). A strong answer traces this to the importance weights themselves: with `q_std=0.5`, the
proposal's tails decay *much faster* than the target's (since $q\sim N(0,0.5)$ has a much smaller
variance than $f\sim N(3,1)$), so the weight $w(x)=f(x)/q(x)$ grows roughly like
$\exp\!\big(x^2(\frac{1}{2\cdot0.5^2}-\frac12)\big)$ for large $x$ — an unbounded, exploding
function of $x$ — which means $\mathbb{E}_q[w(X)^2]$ is **infinite** (the weight distribution has
infinite variance) whenever $q$'s variance is small enough relative to $f$'s (concretely, for
these two Gaussians, whenever $q\_std < 1/\sqrt2\approx0.707$; `q_std=0.5` is on the wrong side of
this threshold, `q_std=2.0` is safely on the right side). A strong answer confirms this
empirically: the maximum raw weight at `q_std=0.5` grows sharply with $n$ (from the hundreds at
$n=10{,}000$ to the tens of thousands at $n=1{,}000{,}000$ in a representative run) rather than
stabilizing, while at `q_std=2.0` it stays essentially flat as $n$ grows. The conclusion: a poor
enough proposal doesn't just inflate the *constant* in front of the usual $1/\sqrt n$ Monte Carlo
rate (as exercise 5's ESS framing might suggest) — it can break the *rate itself*, because the
classical Law-of-Large-Numbers/CLT guarantees underlying $O(1/\sqrt n)$ convergence assume a
finite-variance quantity is being averaged, an assumption a badly-mismatched importance-sampling
proposal can silently violate.

</details>

---

## 📚 Further Reading
- Robert & Casella, *Monte Carlo Statistical Methods*
- Neal, ["MCMC Using Hamiltonian Dynamics"](https://arxiv.org/abs/1206.1901) (2011) — the definitive HMC reference
- Hoffman & Gelman, ["The No-U-Turn Sampler"](https://arxiv.org/abs/1111.4246) (NUTS, 2014) — automates HMC tuning
- Holtzman et al., ["The Curious Case of Neural Text Degeneration"](https://arxiv.org/abs/1904.09751) (nucleus sampling)
- MacKay, *Information Theory, Inference, and Learning Algorithms*, Ch. 29–30 (Monte Carlo Methods)

---

*Next notebook: [Probabilistic Models in Machine Learning](10_probabilistic_models_in_ml.ipynb)*